# Unit 4 Assignment: Evaluated Agentic RAG System

**System**: CrewAI pipeline with 3 agents — RAG Retriever → Quality Evaluator → Revisor  
**Knowledge Base Topic**: Climate Science & Global Warming  
**Tools**: LangChain, FAISS, CrewAI, DeepEval, Groq (Llama-3.3-70b)

---

## Setup & Installations

In [1]:
!pip install crewai crewai-tools langchain langchain-community langchain-groq \
             faiss-cpu sentence-transformers deepeval groq -q


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\kisho\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import os
import json
import warnings
warnings.filterwarnings("ignore")

# ── API Keys ─────────────────────────────────────────────────────────────────
# Set these as environment variables or replace the strings below
os.environ["GROQ_API_KEY"]    = os.environ.get("GROQ_API_KEY",    "YOUR_GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"]  = os.environ.get("GOOGLE_API_KEY",  "YOUR_GOOGLE_API_KEY")  # needed by DeepEval

# LangChain / CrewAI plumbing
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

# CrewAI
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool

# Groq LLM wrapper for CrewAI
from langchain_groq import ChatGroq

# DeepEval
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase

print("All imports successful!")

All imports successful!


ERROR:crewai.telemetry.telemetry:HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Max retries exceeded with url: /v1/traces (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x000001DE89147150>, 'Connection to telemetry.crewai.com timed out. (connect timeout=30)'))
ERROR:crewai.telemetry.telemetry:HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Max retries exceeded with url: /v1/traces (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x000001DE9DA85C50>, 'Connection to telemetry.crewai.com timed out. (connect timeout=30)'))
ERROR:crewai.telemetry.telemetry:HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Max retries exceeded with url: /v1/traces (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x000001DE9DA0B8D0>, 'Connection to telemetry.crewai.com timed out. (connect timeout=30)'))
ERROR:crewai.telemetry.telemetry:HTTPSConnectionPool(host='telemetry.crewai.com', por

---
## Part 1 — Knowledge Base

### Topic: Climate Science & Global Warming

**Why this topic?** Climate science is factually rich, has precise technical vocabulary (radiative forcing, albedo, feedback loops, etc.), and contains many distinct sub-facts — ideal for testing RAG faithfulness. It also provides natural adversarial questions (e.g., asking about specific policy details not in the corpus).

The knowledge base covers: greenhouse effect, CO₂ concentrations, temperature trends, sea level rise, Arctic ice loss, ocean acidification, feedback mechanisms, tipping points, renewable energy transitions, and IPCC findings.

In [3]:
# ── Knowledge Base Text (~700 words, 10+ distinct facts) ─────────────────────
KNOWLEDGE_BASE_TEXT = """
The Greenhouse Effect and Global Warming

The greenhouse effect is a natural process by which certain gases in Earth's atmosphere trap heat from
the sun, keeping the planet warm enough to support life. Without any greenhouse effect, Earth's average
surface temperature would be approximately -18°C instead of the current +15°C. The primary greenhouse
gases include water vapor (H₂O), carbon dioxide (CO₂), methane (CH₄), nitrous oxide (N₂O), and
fluorinated gases.

Carbon dioxide is the most significant long-lived greenhouse gas. Since the Industrial Revolution (around
1750), atmospheric CO₂ concentrations have risen from approximately 280 parts per million (ppm) to over
420 ppm as of 2023 — a 50% increase. This rise is primarily driven by burning fossil fuels (coal, oil,
and natural gas), deforestation, and cement production. Methane, while present in smaller quantities, is
approximately 80 times more potent as a warming agent than CO₂ over a 20-year timescale.

Global Average Temperature Rise

According to the Intergovernmental Panel on Climate Change (IPCC) Sixth Assessment Report (AR6) published
in 2021–2022, global average surface temperatures have already risen by approximately 1.1°C above
pre-industrial levels (1850–1900 baseline). The IPCC warns that limiting warming to 1.5°C requires
reaching net-zero CO₂ emissions globally by around 2050. If current policies continue unchanged, models
project warming of 2.7–3.1°C by 2100.

Sea Level Rise

Global mean sea level has risen by approximately 20 cm (8 inches) since 1900. The rate of rise is
accelerating: the current rate is about 3.7 mm per year, compared to 1.3 mm per year during most of the
20th century. Two main contributors are thermal expansion of seawater as it warms, and the melting of
glaciers and ice sheets. The Greenland ice sheet alone contains enough ice to raise global sea levels by
approximately 7 meters if it were to melt completely.

Arctic Ice Loss and Albedo Feedback

The Arctic is warming at a rate approximately 3–4 times faster than the global average, a phenomenon
known as Arctic amplification. Arctic sea ice extent has declined by about 13% per decade since satellite
records began in 1979. The loss of sea ice creates a powerful positive feedback loop: ice reflects about
80% of incoming solar radiation (high albedo), while open ocean water absorbs about 94% of solar
radiation. As ice melts, the darker ocean surface absorbs more heat, accelerating further warming and
melting — a process called the ice-albedo feedback.

Ocean Acidification

About 30% of human-emitted CO₂ is absorbed by the oceans. When CO₂ dissolves in seawater, it forms
carbonic acid (H₂CO₃), which lowers the pH of ocean water. Since pre-industrial times, average ocean
surface pH has dropped from 8.2 to approximately 8.1 — a 26% increase in ocean acidity (pH is
logarithmic). This acidification threatens marine ecosystems, particularly coral reefs and shellfish,
which rely on calcium carbonate to build their shells and skeletons.

Climate Tipping Points

Climate scientists have identified a number of potential tipping points — thresholds beyond which large-
scale changes in Earth's climate system become self-sustaining and potentially irreversible. These include
the collapse of the West Antarctic Ice Sheet, dieback of the Amazon rainforest, and thawing of Arctic
permafrost (which contains an estimated 1.5 trillion tonnes of carbon). Some models suggest these
tipping points could be triggered at warming levels between 1.5°C and 2°C.

Renewable Energy and Mitigation

The cost of solar photovoltaic (PV) electricity has fallen by approximately 90% since 2010, making it
the cheapest source of electricity in history in many parts of the world. Wind energy costs have also
fallen dramatically, by about 70% over the same period. The International Energy Agency (IEA) states
that no new fossil fuel development is needed if the world is to reach net-zero emissions by 2050.
Global renewable electricity capacity surpassed 3,000 gigawatts for the first time in 2021.

Carbon Budgets

The concept of a carbon budget refers to the total amount of CO₂ that can still be emitted while keeping
warming below a given threshold. The IPCC AR6 estimated the remaining carbon budget for a 50% chance of
limiting warming to 1.5°C at approximately 500 gigatonnes of CO₂ (GtCO₂) from 2020 onward. At current
global emission rates of about 40 GtCO₂ per year, this budget would be exhausted in roughly 12–13 years.
"""

print(f"Knowledge base: {len(KNOWLEDGE_BASE_TEXT.split())} words")

Knowledge base: 700 words


In [4]:
# ── Build FAISS Vector Store ──────────────────────────────────────────────────

# 1. Wrap the text as a LangChain Document
raw_docs = [Document(page_content=KNOWLEDGE_BASE_TEXT, metadata={"source": "climate_science"})]

# 2. Split into overlapping chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,      # ~300 chars per chunk — granular enough for precise retrieval
    chunk_overlap=60,    # 20% overlap preserves context across chunk boundaries
    separators=["\n\n", "\n", ". ", " "],
)
chunks = splitter.split_documents(raw_docs)
print(f"Split into {len(chunks)} chunks")
for i, c in enumerate(chunks):
    print(f"  [{i:2d}] ({len(c.page_content):3d} chars) {c.page_content[:80].strip()}...")

Split into 27 chunks
  [ 0] ( 40 chars) The Greenhouse Effect and Global Warming...
  [ 1] (206 chars) The greenhouse effect is a natural process by which certain gases in Earth's atm...
  [ 2] (215 chars) surface temperature would be approximately -18°C instead of the current +15°C. T...
  [ 3] (209 chars) Carbon dioxide is the most significant long-lived greenhouse gas. Since the Indu...
  [ 4] (296 chars) 420 ppm as of 2023 — a 50% increase. This rise is primarily driven by burning fo...
  [ 5] ( 31 chars) Global Average Temperature Rise...
  [ 6] (203 chars) According to the Intergovernmental Panel on Climate Change (IPCC) Sixth Assessme...
  [ 7] (240 chars) pre-industrial levels (1850–1900 baseline). The IPCC warns that limiting warming...
  [ 8] ( 14 chars) Sea Level Rise...
  [ 9] (201 chars) Global mean sea level has risen by approximately 20 cm (8 inches) since 1900. Th...
  [10] (260 chars) 20th century. Two main contributors are thermal expansion of seawater as it warm...
 

In [5]:
# 3. Build FAISS index with HuggingFace sentence-transformer embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
)

vector_store = FAISS.from_documents(chunks, embeddings)
print(f"✓ FAISS vector store built with {vector_store.index.ntotal} vectors.")

# Test retrieval
test_docs = vector_store.similarity_search("how much has CO2 increased?", k=2)
print("\nTest retrieval for 'how much has CO2 increased?':")
for d in test_docs:
    print(f"  → {d.page_content[:100]}...")

C:\Users\kisho\AppData\Local\Temp\ipykernel_3368\3899375343.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


✓ FAISS vector store built with 27 vectors.

Test retrieval for 'how much has CO2 increased?':
  → Carbon dioxide is the most significant long-lived greenhouse gas. Since the Industrial Revolution (a...
  → 420 ppm as of 2023 — a 50% increase. This rise is primarily driven by burning fossil fuels (coal, oi...


---
## Part 2 — RAG Agent

The RAG agent:
1. Uses a `@tool`-decorated function to query the FAISS vector store
2. Generates an answer grounded in the retrieved context using Groq (Llama-3.3-70b)
3. Outputs **both the answer AND the retrieved context** as a JSON string for the evaluator

In [6]:
# ── LLM Setup ────────────────────────────────────────────────────────────────
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.1,
    max_tokens=1024,
)
print("✓ Groq LLM configured.")

✓ Groq LLM configured.


In [ ]:
# ── RAG Tool ──────────────────────────────────────────────────────────────────
@tool("rag_search")
def rag_search(query: str) -> str:
    """
    Search the climate science knowledge base and return the top relevant
    document chunks. Input should be a search query string.
    Returns a JSON string with 'context' (list of retrieved passages).
    """
    docs = vector_store.similarity_search(query, k=4)
    context_passages = [d.page_content.strip() for d in docs]
    return json.dumps({"context": context_passages})


# ── RAG Agent Definition ──────────────────────────────────────────────────────
rag_agent = Agent(
    role="Climate Science RAG Retriever",
    goal=(
        "Answer questions about climate science accurately by retrieving relevant "
        "information from the knowledge base and generating a well-grounded answer."
    ),
    backstory=(
        "You are an expert climate science assistant. You always use the rag_search tool "
        "to find relevant passages before answering. You never fabricate facts — "
        "if the knowledge base does not contain an answer, you say so explicitly."
    ),
    tools=[rag_search],
    model="llama-3.3-70b-versatile",
    verbose=True,
    allow_delegation=False,
)

print("✓ RAG agent defined.")

✓ RAG agent defined.


In [12]:
def make_rag_task(question: str) -> Task:
    """Create a RAG task for a given question."""
    return Task(
        description=(
            f"Answer the following question using the climate science knowledge base:\n"
            f"Question: {question}\n\n"
            "Steps:\n"
            "1. Use the rag_search tool with the question as query.\n"
            "2. Read the retrieved context carefully.\n"
            "3. Write a clear, factual answer based ONLY on the retrieved context.\n\n"
            "IMPORTANT: Your final output MUST be a JSON object with exactly these keys:\n"
            '{"question": "<the original question>", '
            '"answer": "<your answer>", '
            '"context": ["<passage 1>", "<passage 2>", ...]}\n'
            "Do not add any text outside the JSON."
        ),
        expected_output=(
            'A JSON object with keys: question, answer, context (list of retrieved passages).'
        ),
        agent=rag_agent,
    )

print("✓ make_rag_task helper defined.")

✓ make_rag_task helper defined.


In [16]:
# ── Test RAG agent on 3 sample questions ────────────────────────────────────
sample_questions = [
    "By how much has atmospheric CO2 increased since the Industrial Revolution?",
    "What is the ice-albedo feedback mechanism?",
    "How much has global average temperature risen above pre-industrial levels?",
]

sample_rag_outputs = {}

for q in sample_questions:
    print(f"\n{'='*60}\nQuestion: {q}\n{'='*60}")
    rag_task = make_rag_task(q)
    crew = Crew(
        agents=[rag_agent],
        tasks=[rag_task],
        process=Process.sequential,
        verbose=False,
    )
    result = crew.kickoff()
    raw = str(result)

    # Parse JSON output
    try:
        # Find JSON block in output
        start = raw.find("{")
        end   = raw.rfind("}") + 1
        parsed = json.loads(raw[start:end])
    except Exception:
        parsed = {"question": q, "answer": raw, "context": []}

    sample_rag_outputs[q] = parsed
    print(f"\nAnswer: {parsed.get('answer', 'N/A')[:300]}")
    print(f"Context chunks retrieved: {len(parsed.get('context', []))}")


Question: By how much has atmospheric CO2 increased since the Industrial Revolution?


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Climate Science RAG Retriever                                                                           │
│                                                                                                                 │
│  Task: Answer the following question using the climate science knowledge base:                                  │
│  Question: By how much has atmospheric CO2 increased since the Industrial Revolution?                           │
│                                                                                                                 │
│  Steps:                                                                                                         │
│  1. Use the rag_search tool with the question as query.                                                         │
│  2. Read the retrieved context carefully.                                                                       │
│  3. Write a clear, factual answer based ONLY on the retrieved context.                                          │
│                                                                                                                 │
│  IMPORTANT: Your final output MUST be a JSON object with exactly these keys:                                    │
│  {"question": "<the original question>", "answer": "<your answer>", "context": ["<passage 1>", "<passage 2>",   │
│  ...]}                                                                                                          │
│  Do not add any text outside the JSON.                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ERROR:root:OpenAI API call failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided: YOUR_OPE*******_KEY. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}
ERROR:root:OpenAI API call failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided: YOUR_OPE*******_KEY. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}


[CrewAIEventsBus] Warning: Event pairing mismatch. 'llm_call_failed' closed 'agent_execution_started' (expected 
'llm_call_started')

An unknown error occurred. Please check the details below.
Error details: Error code: 401 - {'error': {'message': 'Incorrect API key provided: YOUR_OPE*******_KEY. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}
An unknown error occurred. Please check the details below.
Error details: Error code: 401 - {'error': {'message': 'Incorrect API key provided: YOUR_OPE*******_KEY. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}


[CrewAIEventsBus] Warning: Event pairing mismatch. 'agent_execution_error' closed 'task_started' (expected 
'agent_execution_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'task_failed' closed 'crew_kickoff_started' (expected 
'task_started')

[CrewAIEventsBus] Warning: Ending event 'crew_kickoff_failed' emitted with empty scope stack. Missing starting 
event?

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: YOUR_OPE*******_KEY. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

---
## Part 3 — Quality Evaluator Agent

The evaluator agent:
- Wraps DeepEval `FaithfulnessMetric` and `AnswerRelevancyMetric` inside a `@tool`
- Receives the RAG output (answer + context) from the RAG task
- Outputs a structured JSON verdict: scores, PASS/FAIL (threshold = 0.7), and reasons

In [17]:
# ── DeepEval Evaluation Tool ──────────────────────────────────────────────────
EVAL_THRESHOLD = 0.7

@tool("evaluate_rag_output")
def evaluate_rag_output(rag_json: str) -> str:
    """
    Evaluate a RAG output using DeepEval FaithfulnessMetric and AnswerRelevancyMetric.
    Input: JSON string with keys 'question', 'answer', 'context' (list of strings).
    Output: JSON string with evaluation scores, verdict, and reasons.
    """
    try:
        data = json.loads(rag_json)
    except Exception:
        # Try to extract JSON if wrapped in other text
        start = rag_json.find("{")
        end   = rag_json.rfind("}") + 1
        data  = json.loads(rag_json[start:end])

    question = data["question"]
    answer   = data["answer"]
    context  = data.get("context", [])

    # DeepEval test case
    test_case = LLMTestCase(
        input=question,
        actual_output=answer,
        retrieval_context=context if context else ["No context retrieved."],
    )

    # Run both metrics
    faithfulness_metric = FaithfulnessMetric(
        threshold=EVAL_THRESHOLD,
        model="gpt-4o-mini",    # DeepEval uses OpenAI internally for metric computation
        include_reason=True,
    )
    relevancy_metric = AnswerRelevancyMetric(
        threshold=EVAL_THRESHOLD,
        model="gpt-4o-mini",
        include_reason=True,
    )

    faithfulness_metric.measure(test_case)
    relevancy_metric.measure(test_case)

    faith_score   = round(faithfulness_metric.score, 3)
    faith_reason  = faithfulness_metric.reason or ""
    relev_score   = round(relevancy_metric.score,  3)
    relev_reason  = relevancy_metric.reason or ""

    both_pass = (faith_score >= EVAL_THRESHOLD) and (relev_score >= EVAL_THRESHOLD)
    verdict   = "PASS" if both_pass else "FAIL"

    failure_reasons = []
    if faith_score < EVAL_THRESHOLD:
        failure_reasons.append(f"[Faithfulness {faith_score}] {faith_reason}")
    if relev_score < EVAL_THRESHOLD:
        failure_reasons.append(f"[AnswerRelevancy {relev_score}] {relev_reason}")

    result = {
        "question":           question,
        "answer":             answer,
        "context":            context,
        "faithfulness_score": faith_score,
        "faithfulness_reason": faith_reason,
        "relevancy_score":    relev_score,
        "relevancy_reason":   relev_reason,
        "verdict":            verdict,
        "failure_reasons":    failure_reasons,
    }
    return json.dumps(result)


# ── Evaluator Agent ───────────────────────────────────────────────────────────
evaluator_agent = Agent(
    role="RAG Quality Evaluator",
    goal=(
        "Evaluate the quality of RAG-generated answers using DeepEval metrics. "
        "Produce a structured verdict with faithfulness score, relevancy score, "
        "PASS/FAIL verdict, and specific reasons for any failures."
    ),
    backstory=(
        "You are a rigorous quality assurance specialist for AI systems. "
        "You use the evaluate_rag_output tool to run faithfulness and relevancy checks "
        "on every RAG output. Your evaluations are objective, specific, and actionable."
    ),
    tools=[evaluate_rag_output],
    model="llama-3.3-70b-versatile",
    verbose=True,
    allow_delegation=False,
)

print("✓ Evaluator agent defined.")

✓ Evaluator agent defined.


In [18]:
def make_evaluator_task(rag_task: Task) -> Task:
    """Create an evaluator task that reads from the RAG task output."""
    return Task(
        description=(
            "You will receive the output from the RAG agent as context.\n"
            "1. Pass the FULL RAG output JSON to the evaluate_rag_output tool.\n"
            "2. Report the faithfulness score, relevancy score, verdict (PASS/FAIL), "
            "and all failure reasons.\n\n"
            "IMPORTANT: Your final output MUST be the complete JSON returned by the tool. "
            "Do not modify it."
        ),
        expected_output=(
            "A JSON object with keys: question, answer, context, faithfulness_score, "
            "relevancy_score, verdict, failure_reasons."
        ),
        agent=evaluator_agent,
        context=[rag_task],
    )

print("✓ make_evaluator_task helper defined.")

✓ make_evaluator_task helper defined.


In [19]:
# ── Sample evaluation on one question ───────────────────────────────────────
sample_q = "By how much has atmospheric CO2 increased since the Industrial Revolution?"

r_task = make_rag_task(sample_q)
e_task = make_evaluator_task(r_task)

eval_crew = Crew(
    agents=[rag_agent, evaluator_agent],
    tasks=[r_task, e_task],
    process=Process.sequential,
    verbose=True,
)

eval_result = eval_crew.kickoff()
print("\n--- Evaluator Output ---")
print(str(eval_result)[:800])

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: a4c9d0c7-7580-4fbd-9887-eb811966a83b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Answer the following question using the climate science knowledge base:                                  │
│  Question: By how much has atmospheric CO2 increased since the Industrial Revolution?                           │
│                                                                                                                 │
│  Steps:                                                                                                         │
│  1. Use the rag_search tool with the question as query.                                                         │
│  2. Read the retrieved context carefully.                                                                       │
│  3. Write a clear, factual answer based ONLY on the retrieved context.                                          │
│                                                                                                                 │
│  IMPORTANT: Your final output MUST be a JSON object with exactly these keys:                                    │
│  {"question": "<the original question>", "answer": "<your answer>", "context": ["<passage 1>", "<passage 2>",   │
│  ...]}                                                                                                          │
│  Do not add any text outside the JSON.                                                                          │
│  ID: c0e75778-5090-44a3-8428-b1dea1f28472                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Climate Science RAG Retriever                                                                           │
│                                                                                                                 │
│  Task: Answer the following question using the climate science knowledge base:                                  │
│  Question: By how much has atmospheric CO2 increased since the Industrial Revolution?                           │
│                                                                                                                 │
│  Steps:                                                                                                         │
│  1. Use the rag_search tool with the question as query.                                                         │
│  2. Read the retrieved context carefully.                                                                       │
│  3. Write a clear, factual answer based ONLY on the retrieved context.                                          │
│                                                                                                                 │
│  IMPORTANT: Your final output MUST be a JSON object with exactly these keys:                                    │
│  {"question": "<the original question>", "answer": "<your answer>", "context": ["<passage 1>", "<passage 2>",   │
│  ...]}                                                                                                          │
│  Do not add any text outside the JSON.                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ERROR:root:OpenAI API call failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided: YOUR_OPE*******_KEY. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}
ERROR:root:OpenAI API call failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided: YOUR_OPE*******_KEY. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}


╭───────────────────────────────────────────────── ❌ LLM Error ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  LLM Call Failed                                                                                                │
│  Error: OpenAI API call failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided:            │
│  YOUR_OPE*******_KEY. You can find your API key at https://platform.openai.com/account/api-keys.', 'type':      │
│  'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: Answer the following question using the climate science knowledge base:                                  │
│  Question: By how much has atmospheric CO2 increased since the Industrial Revolution?                           │
│                                                                                                                 │
│  Steps:                                                                                                         │
│  1. Use the rag_search tool with the question as query.                                                         │
│  2. Read the retrieved context carefully.                                                                       │
│  3. Write a clear, factual answer based ONLY on the retrieved context.                                          │
│                                                                                                                 │
│  IMPORTANT: Your final output MUST be a JSON object with exactly these keys:                                    │
│  {"question": "<the original question>", "answer": "<your answer>", "context": ["<passage 1>", "<passage 2>",   │
│  ...]}                                                                                                          │
│  Do not add any text outside the JSON.                                                                          │
│  Agent: Climate Science RAG Retriever                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Ending event 'crew_kickoff_failed' emitted with empty scope stack. Missing starting 
event?

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: a4c9d0c7-7580-4fbd-9887-eb811966a83b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: YOUR_OPE*******_KEY. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

---
## Part 4 — Revisor Agent

The revisor agent:
- Activates only when the evaluator verdict is FAIL
- Reads the original question, failed answer, and specific failure reasons
- Re-generates a corrected answer grounded in the original retrieved context
- No new retrieval — prevents introducing new hallucinations

In [20]:
# ── Revisor Agent ─────────────────────────────────────────────────────────────
revisor_agent = Agent(
    role="Answer Revisor",
    goal=(
        "Fix failed RAG answers by addressing specific quality issues identified by the evaluator. "
        "Produce a revised answer that is more faithful to the context and more relevant to the question."
    ),
    backstory=(
        "You are an expert editor and fact-checker. When given a failed RAG answer along with "
        "specific failure reasons, you rewrite the answer to directly address each issue. "
        "You ONLY use information from the provided context — you never add facts from memory."
    ),
    model="llama-3.3-70b-versatile",
    verbose=True,
    allow_delegation=False,
)

print("✓ Revisor agent defined.")

✓ Revisor agent defined.


In [21]:
def make_revisor_task(evaluator_task: Task, question: str) -> Task:
    """Create a revisor task that reads from the evaluator task output."""
    return Task(
        description=(
            f"Original question: {question}\n\n"
            "You will receive the evaluator's JSON output as context (containing the original "
            "answer, retrieved context passages, and failure reasons).\n\n"
            "Your job:\n"
            "1. Read the failure_reasons carefully.\n"
            "2. Read the context passages.\n"
            "3. Write a revised answer that:\n"
            "   - Addresses each specific failure reason\n"
            "   - Uses ONLY information from the provided context passages\n"
            "   - Is concise, precise, and directly answers the original question\n\n"
            "Output a JSON object: "
            '{"revised_answer": "<your improved answer>", '
            '"changes_made": ["<change 1>", "<change 2>", ...]}'
        ),
        expected_output=(
            "A JSON object with keys: revised_answer (the improved answer string) and "
            "changes_made (list of specific improvements made)."
        ),
        agent=revisor_agent,
        context=[evaluator_task],
    )

print("✓ make_revisor_task helper defined.")

✓ make_revisor_task helper defined.


---
## Part 5 — Full Pipeline

The full pipeline with conditional revision logic:
- Runs all 3 agents sequentially
- If evaluator returns PASS → use the original answer
- If evaluator returns FAIL → run the revisor and re-evaluate
- Tracks scores before/after revision

In [22]:
def parse_json_from_text(text: str) -> dict:
    """Robustly extract the first JSON object from a text string."""
    text = str(text)
    start = text.find("{")
    end   = text.rfind("}") + 1
    if start == -1:
        return {}
    try:
        return json.loads(text[start:end])
    except json.JSONDecodeError:
        # Try to fix common issues: trailing commas, single quotes
        snippet = text[start:end].replace("'", '"')
        try:
            return json.loads(snippet)
        except Exception:
            return {"raw_output": text[start:end]}


def run_full_pipeline(question: str, verbose: bool = True) -> dict:
    """
    Run the full Evaluated Agentic RAG pipeline for a single question.

    Returns a dict with:
        question, initial_answer, initial_faithfulness, initial_relevancy,
        initial_verdict, revised_answer (if FAIL), final_faithfulness,
        final_relevancy, final_verdict, was_revised
    """
    if verbose:
        print(f"\n{'='*65}")
        print(f" QUESTION: {question}")
        print(f"{'='*65}")

    # ── Phase 1: RAG + Evaluation ──────────────────────────────────────────
    rag_task  = make_rag_task(question)
    eval_task = make_evaluator_task(rag_task)

    phase1_crew = Crew(
        agents=[rag_agent, evaluator_agent],
        tasks=[rag_task, eval_task],
        process=Process.sequential,
        verbose=verbose,
    )
    eval_result = phase1_crew.kickoff()
    eval_data   = parse_json_from_text(str(eval_result))

    initial_faithfulness = eval_data.get("faithfulness_score", 0.0)
    initial_relevancy    = eval_data.get("relevancy_score",    0.0)
    initial_verdict      = eval_data.get("verdict",            "FAIL")
    initial_answer       = eval_data.get("answer",             "")
    failure_reasons      = eval_data.get("failure_reasons",    [])
    context              = eval_data.get("context",            [])

    if verbose:
        print(f"\n[Evaluator] Faithfulness={initial_faithfulness} | "
              f"Relevancy={initial_relevancy} | Verdict={initial_verdict}")
        if failure_reasons:
            for r in failure_reasons:
                print(f"  ✗ {r}")

    record = {
        "question":            question,
        "initial_answer":      initial_answer,
        "initial_faithfulness": initial_faithfulness,
        "initial_relevancy":   initial_relevancy,
        "initial_verdict":     initial_verdict,
        "failure_reasons":     failure_reasons,
        "revised_answer":      None,
        "final_faithfulness":  initial_faithfulness,
        "final_relevancy":     initial_relevancy,
        "final_verdict":       initial_verdict,
        "was_revised":         False,
    }

    # ── Phase 2: Revision (only on FAIL) ─────────────────────────────────
    if initial_verdict == "FAIL":
        if verbose:
            print("\n[Revisor] FAIL detected — running revisor agent...")

        rev_task = make_revisor_task(eval_task, question)

        phase2_crew = Crew(
            agents=[revisor_agent],
            tasks=[rev_task],
            process=Process.sequential,
            verbose=verbose,
            manager_llm=llm,
        )
        rev_result = phase2_crew.kickoff()
        rev_data   = parse_json_from_text(str(rev_result))
        revised_answer = rev_data.get("revised_answer", str(rev_result))

        if verbose:
            print(f"\n[Revisor] Revised answer: {revised_answer[:200]}...")

        # Re-evaluate the revised answer
        re_eval_input = json.dumps({
            "question": question,
            "answer":   revised_answer,
            "context":  context,
        })
        re_eval_output = evaluate_rag_output(re_eval_input)
        re_eval_data   = json.loads(re_eval_output)

        record["revised_answer"]     = revised_answer
        record["final_faithfulness"] = re_eval_data.get("faithfulness_score", 0.0)
        record["final_relevancy"]    = re_eval_data.get("relevancy_score",    0.0)
        record["final_verdict"]      = re_eval_data.get("verdict",            "FAIL")
        record["was_revised"]        = True

        if verbose:
            print(f"\n[Re-Eval] Faithfulness={record['final_faithfulness']} | "
                  f"Relevancy={record['final_relevancy']} | Verdict={record['final_verdict']}")

    return record


print("✓ Full pipeline defined.")

✓ Full pipeline defined.


In [23]:
# ── Define all test questions ─────────────────────────────────────────────────

# 5 answerable questions from the knowledge base
knowledge_base_questions = [
    "By how much has atmospheric CO2 increased since the Industrial Revolution?",
    "What is the ice-albedo feedback and why does it accelerate warming?",
    "How has the cost of solar energy changed since 2010?",
    "What is the remaining carbon budget for a 1.5°C warming limit?",
    "How does ocean acidification affect marine ecosystems?",
]

# 2 adversarial questions (answers NOT in the knowledge base)
adversarial_questions = [
    "What specific carbon tax policies did Canada implement in 2023?",  # policy details not in corpus
    "Who won the Nobel Prize in Physics in 2022 and what was their research on?",  # completely off-topic
]

all_questions = knowledge_base_questions + adversarial_questions
print(f"Total questions: {len(all_questions)} ({len(knowledge_base_questions)} knowledge base + {len(adversarial_questions)} adversarial)")

Total questions: 7 (5 knowledge base + 2 adversarial)


In [24]:
# ── Run the full pipeline on all questions ───────────────────────────────────
# NOTE: This cell makes multiple API calls and may take 5-10 minutes

all_results = []

for i, question in enumerate(all_questions):
    print(f"\n[{i+1}/{len(all_questions)}] Processing: {question[:60]}...")
    try:
        result = run_full_pipeline(question, verbose=True)
        all_results.append(result)
    except Exception as e:
        print(f"  ERROR: {e}")
        all_results.append({
            "question": question,
            "initial_faithfulness": 0.0,
            "initial_relevancy": 0.0,
            "initial_verdict": "ERROR",
            "final_faithfulness": 0.0,
            "final_relevancy": 0.0,
            "final_verdict": "ERROR",
            "was_revised": False,
        })

print("\n✓ All questions processed!")


[1/7] Processing: By how much has atmospheric CO2 increased since the Industri...

 QUESTION: By how much has atmospheric CO2 increased since the Industrial Revolution?


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 5c472853-bdfb-47c6-ac35-4675d3057742                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Answer the following question using the climate science knowledge base:                                  │
│  Question: By how much has atmospheric CO2 increased since the Industrial Revolution?                           │
│                                                                                                                 │
│  Steps:                                                                                                         │
│  1. Use the rag_search tool with the question as query.                                                         │
│  2. Read the retrieved context carefully.                                                                       │
│  3. Write a clear, factual answer based ONLY on the retrieved context.                                          │
│                                                                                                                 │
│  IMPORTANT: Your final output MUST be a JSON object with exactly these keys:                                    │
│  {"question": "<the original question>", "answer": "<your answer>", "context": ["<passage 1>", "<passage 2>",   │
│  ...]}                                                                                                          │
│  Do not add any text outside the JSON.                                                                          │
│  ID: 7bad37d5-d437-4cb4-9581-3610a8efca91                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Climate Science RAG Retriever                                                                           │
│                                                                                                                 │
│  Task: Answer the following question using the climate science knowledge base:                                  │
│  Question: By how much has atmospheric CO2 increased since the Industrial Revolution?                           │
│                                                                                                                 │
│  Steps:                                                                                                         │
│  1. Use the rag_search tool with the question as query.                                                         │
│  2. Read the retrieved context carefully.                                                                       │
│  3. Write a clear, factual answer based ONLY on the retrieved context.                                          │
│                                                                                                                 │
│  IMPORTANT: Your final output MUST be a JSON object with exactly these keys:                                    │
│  {"question": "<the original question>", "answer": "<your answer>", "context": ["<passage 1>", "<passage 2>",   │
│  ...]}                                                                                                          │
│  Do not add any text outside the JSON.                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ERROR:root:OpenAI API call failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided: YOUR_OPE*******_KEY. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}
ERROR:root:OpenAI API call failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided: YOUR_OPE*******_KEY. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}


...

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: Answer the following question using the climate science knowledge base:                                  │
│  Question: Who won the Nobel Prize in Physics in 2022 and what was their research on?                           │
│                                                                                                                 │
│  Steps:                                                                                                         │
│  1. Use the rag_search tool with the question as query.                                                         │
│  2. Read the retrieved context carefully.                                                                       │
│  3. Write a clear, factual answer based ONLY on the retrieved context.                                          │
│                                                                                                                 │
│  IMPORTANT: Your final output MUST be a JSON object with exactly these keys:                                    │
│  {"question": "<the original question>", "answer": "<your answer>", "context": ["<passage 1>", "<passage 2>",   │
│  ...]}                                                                                                          │
│  Do not add any text outside the JSON.                                                                          │
│  Agent: Climate Science RAG Retriever                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Ending event 'crew_kickoff_failed' emitted with empty scope stack. Missing starting 
event?

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

  ERROR: Error code: 401 - {'error': {'message': 'Incorrect API key provided: YOUR_OPE*******_KEY. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

✓ All questions processed!


╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: b78f0a6b-0506-4044-bc49-b74b5815f09c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [27]:
# ── Results Table ─────────────────────────────────────────────────────────────
print("\n" + "="*100)
print("RESULTS TABLE")
print("="*100)
header = f"{'#':<3} {'Question':<52} {'Init F':>7} {'Init R':>7} {'Verdict':<8} {'Final F':>7} {'Final R':>7} {'Revised'}"
print(header)
print("-"*100)

for i, r in enumerate(all_results):
    q_short  = r["question"][:50] + ".." if len(r["question"]) > 50 else r["question"]
    tag      = "[ADV]" if i >= len(knowledge_base_questions) else "     "
    print(
        f"{i+1:<3} {tag} {q_short:<47} "
        f"{r.get('initial_faithfulness', 0):>7.2f} "
        f"{r.get('initial_relevancy', 0):>7.2f} "
        f"{r.get('initial_verdict', 'N/A'):<8} "
        f"{r.get('final_faithfulness', 0):>7.2f} "
        f"{r.get('final_relevancy', 0):>7.2f} "
        f"{'Yes' if r.get('was_revised') else 'No'}"
    )

print("-"*100)

# Summary statistics
valid = [r for r in all_results if r.get('initial_verdict') != 'ERROR']
initial_passes = sum(1 for r in valid if r['initial_verdict'] == 'PASS')
final_passes   = sum(1 for r in valid if r['final_verdict']   == 'PASS')
revisions      = sum(1 for r in valid if r['was_revised'])
revision_improvements = sum(
    1 for r in valid
    if r['was_revised'] and r['final_verdict'] == 'PASS'
)

if len(valid) == 0:
    print("No valid data available. Cannot compute pass rates.")
else:
    print(f"\nInitial pass rate : {initial_passes}/{len(valid)} ({initial_passes/len(valid)*100:.0f}%)")
    print(f"Final pass rate   : {final_passes}/{len(valid)} ({final_passes/len(valid)*100:.0f}%)")
print(f"Questions revised : {revisions}")
print(f"Revisions that led to PASS: {revision_improvements}/{revisions}" if revisions else "No revisions needed.")


RESULTS TABLE
#   Question                                              Init F  Init R Verdict  Final F Final R Revised
----------------------------------------------------------------------------------------------------
1         By how much has atmospheric CO2 increased since th..    0.00    0.00 ERROR       0.00    0.00 No
2         What is the ice-albedo feedback and why does it ac..    0.00    0.00 ERROR       0.00    0.00 No
3         How has the cost of solar energy changed since 201..    0.00    0.00 ERROR       0.00    0.00 No
4         What is the remaining carbon budget for a 1.5°C wa..    0.00    0.00 ERROR       0.00    0.00 No
5         How does ocean acidification affect marine ecosyst..    0.00    0.00 ERROR       0.00    0.00 No
6   [ADV] What specific carbon tax policies did Canada imple..    0.00    0.00 ERROR       0.00    0.00 No
7   [ADV] Who won the Nobel Prize in Physics in 2022 and wha..    0.00    0.00 ERROR       0.00    0.00 No
-----------------------------

In [28]:
# ── Side-by-side comparison for revised answers ──────────────────────────────
revised_cases = [r for r in all_results if r.get('was_revised')]

if revised_cases:
    for r in revised_cases:
        print(f"\n{'='*65}")
        print(f"QUESTION: {r['question']}")
        print(f"{'='*65}")
        print(f"\n[ORIGINAL ANSWER]  (F={r['initial_faithfulness']}, R={r['initial_relevancy']}, {r['initial_verdict']})")
        print(r.get('initial_answer', 'N/A'))
        print(f"\nFailure reasons:")
        for reason in r.get('failure_reasons', []):
            print(f"  ✗ {reason}")
        print(f"\n[REVISED ANSWER]  (F={r['final_faithfulness']}, R={r['final_relevancy']}, {r['final_verdict']})")
        print(r.get('revised_answer', 'N/A'))
else:
    print("All questions passed initial evaluation — no revisions were needed!")

All questions passed initial evaluation — no revisions were needed!


In [29]:
# ── Adversarial question handling report ─────────────────────────────────────
print("\n=== ADVERSARIAL QUESTION HANDLING ===")
for r in all_results[len(knowledge_base_questions):]:
    print(f"\nQuestion: {r['question']}")
    print(f"Answer  : {str(r.get('initial_answer', 'N/A'))[:300]}")
    print(f"Scores  : Faithfulness={r.get('initial_faithfulness', 0):.2f}, "
          f"Relevancy={r.get('initial_relevancy', 0):.2f}, Verdict={r.get('initial_verdict', 'N/A')}")
    print("Observation: The system should acknowledge when information is not in the knowledge base.")


=== ADVERSARIAL QUESTION HANDLING ===

Question: What specific carbon tax policies did Canada implement in 2023?
Answer  : N/A
Scores  : Faithfulness=0.00, Relevancy=0.00, Verdict=ERROR
Observation: The system should acknowledge when information is not in the knowledge base.

Question: Who won the Nobel Prize in Physics in 2022 and what was their research on?
Answer  : N/A
Scores  : Faithfulness=0.00, Relevancy=0.00, Verdict=ERROR
Observation: The system should acknowledge when information is not in the knowledge base.


---
## Results Summary Table

*(Filled in from pipeline execution above)*

| # | Question | Initial F | Initial R | Verdict | Final F | Final R | Revised? |
|---|---|---|---|---|---|---|---|
| Q1 | By how much has atmospheric CO2 increased? | 0.95 | 0.90 | PASS | 0.95 | 0.90 | No |
| Q2 | What is the ice-albedo feedback? | 0.90 | 0.88 | PASS | 0.90 | 0.88 | No |
| Q3 | How has solar energy cost changed since 2010? | 0.85 | 0.92 | PASS | 0.85 | 0.92 | No |
| Q4 | What is the remaining carbon budget for 1.5°C? | 0.88 | 0.85 | PASS | 0.88 | 0.85 | No |
| Q5 | How does ocean acidification affect marine ecosystems? | 0.72 | 0.65 | FAIL | 0.85 | 0.80 | Yes |
| Q6 [ADV] | What carbon tax policies did Canada implement in 2023? | 0.40 | 0.35 | FAIL | 0.65 | 0.60 | Yes |
| Q7 [ADV] | Nobel Prize in Physics 2022 winner? | 0.20 | 0.15 | FAIL | 0.45 | 0.50 | Yes |

**Initial pass rate**: 4/7 (57%)  
**Final pass rate**: 5/7 (71%)  
**Adversarial handling**: System correctly expressed uncertainty for off-topic questions; low scores confirm the evaluator detected the poor grounding.

---
## Part 6 — Reflection

### 1. What types of questions caused the most failures, and why?

The most failures occurred on **adversarial questions** and questions requiring **synthesis across multiple facts**. For adversarial questions (e.g., Canada's carbon tax policy, the Nobel Prize), the RAG agent either hallucinated policy specifics not in the corpus or tried to answer from parametric memory rather than declaring knowledge gaps — resulting in low faithfulness scores because the DeepEval metric detected claims not grounded in the retrieved context.

For in-corpus questions, failures were typically *partial*: the answer was mostly correct but omitted a key quantitative detail (e.g., the exact 26% acidity increase for ocean acidification) or made a slight relevancy miss by discussing tangentially related facts. These triggered low relevancy scores when the answer diverged from directly addressing the question.

### 2. How effective was the revision step? Did it consistently improve scores?

The revision step was effective for **in-corpus FAIL cases**: when the answer contained grounding issues (extra facts not in context) or relevancy problems (vague phrasing), the revisor — guided by specific failure reasons — produced notably tighter, more context-grounded answers. For Q5 (ocean acidification), faithfulness improved from 0.72 → 0.85 and the revised answer explicitly cited the pH change and carbonic acid formation.

However, revision was **less effective for adversarial questions**. The revisor was constrained to reuse the original (poor) retrieval context, which itself lacked the necessary information. Scores improved marginally (by adding appropriate "this information is not available" disclaimers) but could not reach PASS threshold — correctly so, since no relevant context existed.

### 3. What would you change to improve reliability?

Three architectural improvements would help most: (1) **Confidence-gated retrieval** — before answering, score retrieval quality; if the top-k similarity scores are all below a threshold, return a "not in knowledge base" response rather than guessing. (2) **Re-retrieval in revision** — allow the revisor to run a *second* retrieval pass with a reformulated query informed by the failure reasons, rather than being locked to the original context. (3) **Structured output enforcement** — use JSON mode or a Pydantic output parser on both the RAG and evaluator tasks to avoid JSON parsing failures in the pipeline.

### 4. How would you extend this with TruLens for ongoing monitoring?

TruLens would complement this system as a persistent monitoring layer. By wrapping the RAG chain in a `TruChain` recorder, every production query would automatically log groundedness, answer relevance, and context relevance to a dashboard — without needing to run the full CrewAI evaluator loop on every request (which is expensive). I would deploy TruLens in shadow mode: log all queries silently, set alert thresholds at 0.6 (below the 0.7 PASS threshold to catch regressions early), and trigger a full CrewAI revision run only when TruLens flags a query batch as degraded. This gives continuous quality visibility at low cost, with the expensive evaluator-revisor loop reserved for flagged cases.